# 非凸性（Non-convexity）

## "凸" vs "非凸"：山谷 vs 山地
### 凸函数（Convex）
<font color='red'>想象一个光滑的碗</font>：

In [ ]:
        \       /
         \     /
          \   /
           \_/

- <font color='red'>无论从哪出发，沿着梯度下降，最终都会滑到唯一的最低点</font>
- 任意两点连线都在函数图像上方
- 局部最优 = 全局最优，**没有"陷阱"**
例子：$f(x)=x^2$，线性回归的损失函数

### 非凸函数（Non-convex）
<font color='red'>想象一片崎岖的山地：</font>

In [ ]:
      /\        /\
     /  \  /\  /  \
    /    \/  \/    \____
   /                    \___

- 到处都是局部最低点（小坑）
- **梯度方向可能把你引向一个浅坑，而不是最深的山谷**
- 两点之间可能有悬崖和屏障

# <font color='red'>LLM 策略空间为什么是非凸的？</font>
LLM 的策略是一个概率分布 $π_θ(a∣s)$ ，参数 θ  是高维空间（数十亿维）中的一个点。**这个空间极度非凸，原因有三：**

##  参数空间的复杂性
Transformer 的注意力机制、非线性激活（GELU/SiLU）、层与层之间的耦合，**使得损失函数 landscape 像山地一样崎岖。**

In [ ]:
损失
 ↑    ●●●
 │   ●   ●        ← 一个局部最优（模型学会某种固定模式）
 │  ●     ●●●
 │ ●           ●●●  ← 另一个局部最优（完全不同的生成风格）
 │●    ●●●    ●
 └────────────────→ 参数变化方向

## 离散输出的不可导性
LLM 输出的是离散 token（从词汇表中采样）。策略梯度通过 log-probability 的梯度来优化，**但这个梯度 landscape 对参数变化极其敏感：**
- 稍微改变某个注意力头的权重 → 某个 token 的概率从 0.1% 跳到 50%
- 损失函数在这个区域是不连续、高度非线性的


## RL 的反馈回路加剧了非凸性
在监督学习中，数据是固定的；<font color='red'>在 RL 中，数据分布取决于当前策略：</font>

In [ ]:
策略变一点 → 生成的序列大变 → Reward 计算大变 → 梯度方向大变

这相当于"地形本身在随你的位置变化"，让优化 landscape 更加崎岖。

# 非凸性带来的实际问题

| 问题        | 直观解释                            |
| :-------- | :------------------------------ |
| **梯度不可靠** | 当前梯度指向的"下坡"方向，可能通向一个浅坑而非全局最优    |
| **初始化敏感** | 同样的算法，不同的随机种子，可能收敛到完全不同的模型      |
| **模式崩溃**  | 模型可能掉进一个"局部最优陷阱"——比如只学会重复某个高分短语 |
| **训练不稳定** | 一步大的更新可能从"好区域"跳到"坏区域"，再也爬不回来    |


<font color='red'>这正是为什么 RL 训练中需要：</font>
- 梯度裁剪（防止一步跨太大，掉进悬崖）
- PPO 的 ratio clip（限制单次更新幅度）
- **小学习率 + 大量步骤（小心翼翼地摸索下山）**

# 一句话总结
- 非凸性意味着 LLM 的策略优化空间不是"一个光滑的碗"，而是"一片布满陷阱和悬崖的崎岖山地"。
- <font color='red'>梯度只是告诉你"脚下最陡的方向"，但这个方向未必通向真正的最优解</font>——它可能只是旁边一个小坑的入口(局部最优解)。